In [12]:
!pip install ipython FinMind matplotlib pandas
!pip install pandas requests matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 下載繁體中文字型
!wget -O SourceHanSerifTW-VF.ttf https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf

# 加入字型檔
fm.fontManager.addfont('SourceHanSerifTW-VF.ttf')

# 設定字型
#
mpl.rc('font', family='Source Han Serif TW VF')

--2026-06-25 07:23:19--  https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/adobe-fonts/source-han-serif/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf [following]
--2026-06-25 07:23:19--  https://raw.githubusercontent.com/adobe-fonts/source-han-serif/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 

185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16851180 (16M) [application/octet-stream]
Saving to: ‘SourceHanSerifTW-VF.ttf’

SourceHanSerifTW-VF 100%[===================>]  16.07M  --.-KB/s    in 0.1s    

2026-06-25 07:23:19 (155 MB/s) - ‘SourceHanSerifTW-VF.ttf’ saved [16851180/16851180]



In [14]:
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import pandas as pd
import requests

# ==================== 1. 字型下載與系統全域設定 ====================
# 下載繁體中文字型 (思源宋體)
!wget -O SourceHanSerifTW-VF.ttf https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf

# 將字型檔加入字型管理員
fm.fontManager.addfont('SourceHanSerifTW-VF.ttf')

# 全域設定字型與負號顯示
mpl.rc('font', family='Source Han Serif TW VF')
plt.rcParams['axes.unicode_minus'] = False


# ==================== 2. 通用資料獲取函式 (透過 API 規避套件 Bug) ====================
def fetch_stock_annual_data_api(stock_id, start_date="2001-01-01", end_date="2025-12-31"):
    """
    透過標準 HTTP 請求至 FinMind 數據伺服器，自動清洗並轉換為年度資料。
    """
    api_url = "https://api.finmindtrade.com/api/v4/data"
    
    # A. 抓取綜合損益表資料
    payload_fin = {
        "dataset": "TaiwanStockFinancialStatement",
        "data_id": stock_id,
        "start_date": start_date,
        "end_date": end_date
    }
    res_fin = requests.get(api_url, params=payload_fin).json()
    df_financial = pd.DataFrame(res_fin.get('data', []))
    
    # B. 抓取每日技術面股價資料
    payload_price = {
        "dataset": "TaiwanStockPrice",
        "data_id": stock_id,
        "start_date": start_date,
        "end_date": end_date
    }
    res_price = requests.get(api_url, params=payload_price).json()
    df_price = pd.DataFrame(res_price.get('data', []))
    
    # 檢查是否成功取得資料
    if df_financial.empty or df_price.empty:
        print(f"⚠️ 股票代號 {stock_id} 未能自伺服器取得完整財報或股價資料。")
        return pd.DataFrame()

    # 統一將欄位名稱調整為小寫，提升代碼穩健度
    df_financial.columns = df_financial.columns.str.lower()
    df_price.columns = df_price.columns.str.lower()

    # C. 資料清洗：EPS 季度加總轉年度
    df_eps = df_financial[df_financial['type'].str.contains('earning|eps|每股盈餘', case=False, na=False)].copy()
    df_eps['date'] = pd.to_datetime(df_eps['date'])
    df_eps['year'] = df_eps['date'].dt.year
    df_annual_eps = df_eps.groupby('year')['value'].sum().reset_index().rename(columns={'value': 'eps'})

    # D. 資料清洗：股價日資料轉年度平均
    df_price['date'] = pd.to_datetime(df_price['date'])
    df_price['year'] = df_price['date'].dt.year
    df_annual_price = df_price.groupby('year')['close'].mean().reset_index().rename(columns={'close': 'price'})

    # 合併兩者回傳
    return pd.merge(df_annual_eps, df_annual_price, on='year', how='inner')


# ==================== 3. 動態多子圖參數化繪製函式 ====================
def plot_stock_comparison(stock_dict, start_date="2010-01-01", end_date="2025-12-31"):
    """
    依據傳入的字典動態生成多子圖對照。
    :param stock_dict: 字典格式，如 {"代號": "名稱"}
    """
    num_stocks = len(stock_dict)
    if num_stocks == 0:
        print("未偵測到指定的股票清單。")
        return
        
    # 每張子圖動態配置 8 吋寬度，確保排版不擠壓
    fig, axes = plt.subplots(1, num_stocks, figsize=(8 * num_stocks, 6), squeeze=False)
    axes_flat = axes.flatten()
    
    color_price = '#3A6BB7'  # 藍色 (股價)
    color_eps = '#E57D32'    # 橘色 (EPS)
    
    # 走訪字典項目，非寫死變數
    for idx, (stock_id, stock_name) in enumerate(stock_dict.items()):
        print(f"正在遠端獲取 ({stock_id}) {stock_name} 的歷史數據...")
        df_stock = fetch_stock_annual_data_api(stock_id, start_date, end_date)
        
        if df_stock.empty:
            continue
            
        ax_left = axes_flat[idx]
        
        # ---- 左軸 (ax_left)：股價折線圖 ----
        ax_left.plot(df_stock['year'], df_stock['price'], color=color_price, linewidth=3, marker='o', label='股價')
        ax_left.set_ylabel('年度平均股價 (元)', color=color_price, fontsize=11)
        ax_left.tick_params(axis='y', labelcolor=color_price)
        ax_left.set_ylim(0, df_stock['price'].max() * 1.2)
        
        # ---- 右軸 (ax_right)：EPS 長條圖 ----
        ax_right = ax_left.twinx()
        ax_right.bar(df_stock['year'], df_stock['eps'], color=color_eps, width=0.4, alpha=0.9, label='EPS')
        ax_right.set_ylabel('EPS (元)', color=color_eps, fontsize=11)
        ax_right.tick_params(axis='y', labelcolor=color_eps)
        ax_right.set_ylim(0, df_stock['eps'].max() * 1.2)
        
        # ---- 細節外觀與格線修飾 ----
        # 橫軸年份間隔 2 年顯示，防文字重疊
        ax_left.set_xticks(df_stock['year'][::2])
        ax_left.set_xticklabels(df_stock['year'][::2], rotation=45, fontsize=10)
        
        # 標題與邊框去化
        ax_left.set_title(f'{stock_name} ({stock_id})\n歷年 EPS 與股價對照', fontsize=14, fontweight='bold', pad=10)
        ax_left.spines['top'].set_visible(False)
        ax_right.spines['top'].set_visible(False)
        
        # 強制將折線圖拉到最上層，防止長條圖色塊遮擋線條
        ax_left.set_zorder(ax_right.get_zorder() + 1)
        ax_left.patch.set_visible(False)

    # ==================== 4. 公用圖例與總圖標設定 ====================
    lines_l, labels_l = ax_left.get_legend_handles_labels()
    lines_r, labels_r = ax_right.get_legend_handles_labels()
    fig.legend(lines_l + lines_r, labels_l + labels_r, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False, fontsize=12)
    
    fig.suptitle('長期獲利能力與股價走勢橫向對照', fontsize=18, fontweight='bold', y=1.05)
    
    plt.tight_layout()
    plt.show()


# ==================== 5. 測試執行進入點 ====================
if __name__ == "__main__":
    # 在這裡您可以自由增減、更換想要比較的台股標的
    target_comparison = {
        "2330": "台積電",
        "2412": "中華電信",
        "2454": "聯發科"
    }
    
    # 執行繪圖 (設定觀測區間為 2010 年至 2025 年)
    plot_stock_comparison(stock_dict=target_comparison, start_date="2010-01-01", end_date="2025-12-31")

--2026-06-25 07:26:17--  https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/adobe-fonts/source-han-serif/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf [following]
--2026-06-25 07:26:17--  https://raw.githubusercontent.com/adobe-fonts/source-han-serif/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16851180 (16M) [application/octet-stream]
Saving to: ‘SourceHanSerifTW-VF.ttf’

SourceHanSerifTW-VF 100%[===================>]  16.07M  --.-

ModuleNotFoundError: No module named 'IPython.core.pylabtools'

In [16]:
import os
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import pandas as pd
import requests

# ==================== 1. 字型下載與系統全域設定 (Codespaces 相容版) ====================
font_filename = 'SourceHanSerifTW-VF.ttf'

# 檢查字型是否已存在，若不存在則下載 (改用 os.system 避開 !wget 錯誤)
if not os.path.exists(font_filename):
    print("正在下載思源宋體...")
    os.system(f"wget -O {font_filename} https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf")

# 將字型檔加入字型管理員
fm.fontManager.addfont(font_filename)

# 全域設定字型與負號顯示
mpl.rc('font', family='Source Han Serif TW VF')
plt.rcParams['axes.unicode_minus'] = False


# ==================== 2. 通用資料獲取函式 ====================
def fetch_stock_annual_data_api(stock_id, start_date="2001-01-01", end_date="2025-12-31"):
    api_url = "https://api.finmindtrade.com/api/v4/data"
    
    # A. 抓取綜合損益表資料
    payload_fin = {
        "dataset": "TaiwanStockFinancialStatement",
        "data_id": stock_id,
        "start_date": start_date,
        "end_date": end_date
    }
    res_fin = requests.get(api_url, params=payload_fin).json()
    df_financial = pd.DataFrame(res_fin.get('data', []))
    
    # B. 抓取每日技術面股價資料
    payload_price = {
        "dataset": "TaiwanStockPrice",
        "data_id": stock_id,
        "start_date": start_date,
        "end_date": end_date
    }
    res_price = requests.get(api_url, params=payload_price).json()
    df_price = pd.DataFrame(res_price.get('data', []))
    
    if df_financial.empty or df_price.empty:
        print(f"⚠️ 股票代號 {stock_id} 未能取得完整財報或股價資料。")
        return pd.DataFrame()

    df_financial.columns = df_financial.columns.str.lower()
    df_price.columns = df_price.columns.str.lower()

    # C. 資料清洗：EPS 季度加總轉年度
    df_eps = df_financial[df_financial['type'].str.contains('earning|eps|每股盈餘', case=False, na=False)].copy()
    df_eps['date'] = pd.to_datetime(df_eps['date'])
    df_eps['year'] = df_eps['date'].dt.year
    df_annual_eps = df_eps.groupby('year')['value'].sum().reset_index().rename(columns={'value': 'eps'})

    # D. 資料清洗：股價日資料轉年度平均
    df_price['date'] = pd.to_datetime(df_price['date'])
    df_price['year'] = df_price['date'].dt.year
    df_annual_price = df_price.groupby('year')['close'].mean().reset_index().rename(columns={'close': 'price'})

    return pd.merge(df_annual_eps, df_annual_price, on='year', how='inner')


# ==================== 3. 動態多子圖參數化繪製函式 ====================
def plot_stock_comparison(stock_dict, start_date="2010-01-01", end_date="2025-12-31", output_filename="stock_comparison.png"):
    num_stocks = len(stock_dict)
    if num_stocks == 0:
        print("未偵測到指定的股票清單。")
        return
        
    fig, axes = plt.subplots(1, num_stocks, figsize=(8 * num_stocks, 6), squeeze=False)
    axes_flat = axes.flatten()
    
    color_price = '#3A6BB7'  # 藍色 (股價)
    color_eps = '#E57D32'    # 橘色 (EPS)
    
    for idx, (stock_id, stock_name) in enumerate(stock_dict.items()):
        print(f"正在遠端獲取 ({stock_id}) {stock_name} 的歷史數據...")
        df_stock = fetch_stock_annual_data_api(stock_id, start_date, end_date)
        
        if df_stock.empty:
            continue
            
        ax_left = axes_flat[idx]
        
        # ---- 左軸：股價折線圖 ----
        ax_left.plot(df_stock['year'], df_stock['price'], color=color_price, linewidth=3, marker='o', label='股價')
        ax_left.set_ylabel('年度平均股價 (元)', color=color_price, fontsize=11)
        ax_left.tick_params(axis='y', labelcolor=color_price)
        ax_left.set_ylim(0, df_stock['price'].max() * 1.2)
        
        # ---- 右軸：EPS 長條圖 ----
        ax_right = ax_left.twinx()
        ax_right.bar(df_stock['year'], df_stock['eps'], color=color_eps, width=0.4, alpha=0.9, label='EPS')
        ax_right.set_ylabel('EPS (元)', color=color_eps, fontsize=11)
        ax_right.tick_params(axis='y', labelcolor=color_eps)
        ax_right.set_ylim(0, df_stock['eps'].max() * 1.2)
        
        # ---- 細節外觀與格線修飾 ----
        ax_left.set_xticks(df_stock['year'][::2])
        ax_left.set_xticklabels(df_stock['year'][::2], rotation=45, fontsize=10)
        
        ax_left.set_title(f'{stock_name} ({stock_id})\n歷年 EPS 與股價對照', fontsize=14, fontweight='bold', pad=10)
        ax_left.spines['top'].set_visible(False)
        ax_right.spines['top'].set_visible(False)
        
        ax_left.set_zorder(ax_right.get_zorder() + 1)
        ax_left.patch.set_visible(False)

    # ==================== 4. 公用圖例與總圖標設定 ====================
    lines_l, labels_l = ax_left.get_legend_handles_labels()
    lines_r, labels_r = ax_right.get_legend_handles_labels()
    fig.legend(lines_l + lines_r, labels_l + labels_r, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False, fontsize=12)
    
    fig.suptitle('長期獲利能力與股價走勢橫向對照', fontsize=18, fontweight='bold', y=1.05)
    
    plt.tight_layout()
    
    # ⚠️ Codespaces 關鍵：不使用 plt.show()，直接儲存為實體圖片檔
    plt.savefig(output_filename, bbox_inches='tight', dpi=150)
    print(f"\n🎉 圖表已成功儲存！請至左側檔案瀏覽器點擊查看：{output_filename}")
    plt.close()


# ==================== 5. 測試執行進入點 ====================
if __name__ == "__main__":
    target_comparison = {
        "2330": "台積電",
        "2412": "中華電信",
        "2454": "聯發科"
    }
    
    plot_stock_comparison(
        stock_dict=target_comparison, 
        start_date="2010-01-01", 
        end_date="2025-12-31",
        output_filename="stock_comparison.png" # 輸出的圖片檔名
    )

ModuleNotFoundError: No module named 'IPython.core.pylabtools'